# Analisis Geoespacial de Hotspots Comerciales: Wendy's San Salvador
Prototipo academico de Machine Learning (K-Means + KNN con distancia Haversine).

In [ ]:
import pandas as pd
import numpy as np
import folium
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import LeaveOneOut

df = pd.read_csv('../wendys_san_salvador_limpio.csv')
print(f'Total de sucursales cargadas: {len(df)}')
df.head()

## 1. Agrupamiento Espacial con K-Means
Evaluacion de Inercia y Coeficiente de Silueta para identificar polos comerciales.

In [ ]:
X = df[['latitud', 'longitud']].to_numpy()
for k in range(2, min(7, len(df))):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    print(f'K={k} | Inercia={km.inertia_:.6f} | Silueta={silhouette_score(X, labels):.4f}')

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X)
conteo = df['cluster'].value_counts()
promedio = conteo.mean()
df['categoria_densidad'] = np.where(df['cluster'].map(conteo) > promedio, 'Alta Concentracion', 'Baja Concentracion')
df[['nombre', 'latitud', 'longitud', 'cluster', 'categoria_densidad']]

## 2. Clasificacion Supervisada con KNN Haversine y Validacion Leave-One-Out

In [ ]:
X_rad = np.radians(df[['latitud', 'longitud']].to_numpy())
y = np.array(df['categoria_densidad'].tolist())
loo = LeaveOneOut()

for k in range(1, len(df), 2):
    aciertos = 0
    for train_idx, test_idx in loo.split(X_rad):
        knn = KNeighborsClassifier(n_neighbors=k, metric='haversine')
        knn.fit(X_rad[train_idx], y[train_idx])
        if knn.predict(X_rad[test_idx])[0] == y[test_idx][0]:
            aciertos += 1
    print(f'k={k} | Exactitud LOO: {aciertos/len(df):.2%}')

## 3. Evaluacion de Nueva Ubicacion Propuesta

In [ ]:
knn_final = KNeighborsClassifier(n_neighbors=3, metric='haversine')
knn_final.fit(X_rad, y)

punto_nuevo = np.radians([[13.7015, -89.231]])
pred = knn_final.predict(punto_nuevo)[0]
dist_rad, indices = knn_final.kneighbors(punto_nuevo)
dist_km = dist_rad[0] * 6371.0

print('Prediccion:', pred)
for idx, d in zip(indices[0], dist_km):
    print(f"{df.loc[idx, 'nombre']} -> {d:.3f} km")